In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path
from urllib.robotparser import RobotFileParser
from datetime import datetime
import time
import matplotlib.pyplot as plt
import numpy as np

In [15]:
url = "https://bouldercounty.gov/property-and-land/public-trustee/foreclosure/search/"

headers = {
    "User-Agent": "Foreclosure-Value-Search-Project-INFO4617/1.0 (contact: arsh9701@colorado.edu)"
}

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok = True)


In [19]:
if not allowed:
    raise PermissionError("Scraping is not allowed")

time.sleep(3)

response = requests.get(
    url,
    headers = headers,
    timeout=30
)

print("Status:", response.status_code)
print("Final URL:", response.url)

response.raise_for_status()
      

Status: 200
Final URL: https://bouldercounty.gov/property-and-land/public-trustee/foreclosure/search/


In [20]:
robots_url = "https://bouldercounty.gov/robots.txt"

rp = RobotFileParser()
rp.set_url(robots_url)
rp.read()

allowed = rp.can_fetch(headers["User-Agent"], url)

print("Scraping allowed:", allowed)

#lets goooooooo

raw_file = raw_dir / "forclosure_page.html"

raw_file.write_text(
    response.text,
    encoding="utf-8"

)

print("Raw HTML saved:", raw_file)

Scraping allowed: True
Raw HTML saved: data/raw/forclosure_page.html


In [21]:
soup = BeautifulSoup(response.text, "html.parser")

tables = soup.find_all("table")

print("Number of tables:", len(tables))

for i, table in enumerate(tables):
    print(f"Table {i}:")
    print(table.get_text(" ", strip=True)[:500])

Number of tables: 0


In [22]:
#no cells that we can see in HTML, checking JavaScript

print("Status Code:", response.status_code)
print("Final URL:", response.url)
print("Page Title:", soup.title.get_text(strip=True) if soup.title else "No title")
print("HTML Length:", len(response.text))

print("/nFirst 1000 characters:")
print(response.text[:1000])

Status Code: 200
Final URL: https://bouldercounty.gov/property-and-land/public-trustee/foreclosure/search/
Page Title: Foreclosure Search - Boulder County
HTML Length: 161661
/nFirst 1000 characters:
<!doctype html>
<html lang="en-US">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<!-- Mobile Specific Metas & Favicons -->
<meta name="viewport" content="width=device-width, initial-scale=1"><link rel="shortcut icon" href="https://assets.bouldercounty.gov/wp-content/uploads/2017/01/logo_boco3.png"><link rel="apple-touch-icon" href="https://assets.bouldercounty.gov/wp-content/uploads/2018/07/BOCOlogo_57.png"><link rel="apple-touch-icon" sizes="120x120" href="https://assets.bouldercounty.gov/wp-content/uploads/2018/07/BOCOlogo_120.png"><link rel="apple-touch-icon" sizes="76x76" href="https://assets.bouldercounty.gov/wp-content/uploads/2018/07/BOCOlogo_76.png"><link rel="apple-touch-icon" sizes="152x152" href="https://assets.bouldercounty.gov/wp-content/uploads

In [23]:
links = soup.find_all("a", href=True)

print("Total links:", len(links))

for link in links[:20]:
    print(link.get_text(" ", strip=True), link["href"])

Total links: 402
Skip to menu #navigation
Skip to content #content
Skip to footer #footer
Property Search https://maps.bouldercounty.org/boco/PropertySearch/
Jobs https://bouldercounty.gov/jobs/
News /news
Elections https://bouldercounty.gov/elections/
County Government https://bouldercounty.gov/government/
Departments https://bouldercounty.gov/departments/
Contact https://bouldercounty.gov/government/contact-us/
 https://bouldercounty.gov/
 #
Community Support & Housing https://bouldercounty.gov/families/
Adults and Seniors https://bouldercounty.gov/families/seniors/
Adult Protective Services https://bouldercounty.gov/families/seniors/adult-protection-senior-services/
Boulder County Area Agency on Aging https://bouldercounty.gov/departments/community-services/area-agency-aging/
Community Action Programs https://bouldercounty.gov/departments/community-services/community-action-programs/
Long-Term Services and Supports https://bouldercounty.gov/families/seniors/long-term-care/
Old Age P

In [24]:
print("Status code:", response.status_code)
print("Page Title:", soup.title.get_text(strip=True))
print("Number of tables:", len(soup.find_all("table")))
print("Number of iframes:", len(soup.find_all("iframe")))

Status code: 200
Page Title: Foreclosure Search - Boulder County
Number of tables: 0
Number of iframes: 1


In [25]:
soup = BeautifulSoup(response.text, "html.parser")

print("Status code:", response.status_code)
print("Page Title:", soup.title.get_text(strip=True) if soup.title else "No title")
print("Number of tables:", len(soup.find_all("table")))
print("Number of iframes:", len(soup.find_all("iframe")))

Status code: 200
Page Title: Foreclosure Search - Boulder County
Number of tables: 0
Number of iframes: 1


In [26]:
iframe = soup.find("iframe")

print("Iframe source:", iframe.get("src"))

Iframe source: https://www.bouldercountypt.org/GTSSearch


In [28]:
search_url = "https://www.bouldercountypt.org/GTSSearch"

time.sleep(3)

search_response = requests.get(
    search_url,
    headers=headers,
    timeout=30
)

print("Status Code:", search_response.status_code)
print("Final URL:", search_response.url)

search_response.raise_for_status()

search_soup = BeautifulSoup(
    search_response.text,
    "html.parser"
)

print("Page Title:",
      search_soup.title.get_text(strip=True)
      if search_soup.title else "No title")

print("Number of tables:",
      len(search_soup.find_all("table")))

print("Number of forms:",
      len(search_soup.find_all("form")))

Status Code: 200
Final URL: https://bouldercountypt.org/GTSSearch/(S(cvut2fkekwazc5phbqhughqg))/default
Page Title: Boulder Foreclosure Search
Number of tables: 0
Number of forms: 1


In [29]:
for form in search_soup.find_all("form"):
    print("Form action:", form.get("action"))
    print("Form method:", form.get("method"))

    for field in form.find_all(["input", "select"]):
        print(
            "Field:",
            field.get("name"),
            "| Type:",
            field.get("type")
        )
        

Form action: ./default
Form method: post
Field: __VIEWSTATE | Type: hidden
Field: __VIEWSTATEGENERATOR | Type: hidden
Field: __EVENTVALIDATION | Type: hidden
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtForeclosureNumber | Type: text
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtGrantorsName | Type: text
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtOwersName | Type: text
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtZipCode | Type: text
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtStreet | Type: text
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtSubdivision | Type: text
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$ddStatus | Type: None
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtNedDate1 | Type: date
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtNedDate2 | Type: date
Field: ctl00$ctl00$MainContent$CustomContentPlaceHolder$txtSoldDate1 | Type: date
Field: ctl00$ctl00$M

In [31]:
for button in search_soup.find_all(
    ["input", "button"]
):
    if button.get("type") in ["submit", "button"]:
        print(
            "Name:", button.get("name"),
            "| Value:", button.get("value"),
            "| ID:", button.get("id")
        )

        print(
        "/nCAPTCHA elements:",
        len(search_soup.select(".g-recaptcha"))
        )

        print(
            "Tables:",
            len(search_soup.find_all("table"))
        )
        

Name: ctl00$ctl00$MainContent$CustomContentPlaceHolder$btnReset | Value: Reset | ID: MainContent_CustomContentPlaceHolder_btnReset
/nCAPTCHA elements: 0
Tables: 0
Name: ctl00$ctl00$MainContent$CustomContentPlaceHolder$btnSearch | Value: Search | ID: MainContent_CustomContentPlaceHolder_btnSearch
/nCAPTCHA elements: 0
Tables: 0
Name: ctl00$ctl00$MainContent$CustomContentPlaceHolder$btnShowAll | Value: Show All | ID: MainContent_CustomContentPlaceHolder_btnShowAll
/nCAPTCHA elements: 0
Tables: 0
